# Differential spatial patterns {#sec-mult-diff-spatial-patterns}



## Preamble

### Introduction

Recent advances in spatial transcriptomics (ST) techniques have facilitated the generation of multi-sample datasets across various experimental conditions (e.g. healthy vs. diseased, or multiple treatments). When multi-sample, multi-condition ST data are available, differential analysis can be performed to identify genes with distinct spatial expression patterns that change between conditions either across domains or within any domain; we call these **differential spatial pattern (DSP)** genes. 

A DSP gene may be spatially variable in one condition and display spatially uniform expression in another one, or it could be spatially variable in all conditions, but with distinct spatial expression patterns. Conversely, genes that show spatially uniform abundance, or that are spatially variable with the same spatial structure in all conditions, are not considered DSP, because their spatial distribution is unchanged. Another way to think about DSP is that it is the spatial analog of 'differential state analysis' for single cell analyses. That is, here we focus on differences in expression within any domain (or different across domains) between conditions.

Identifying DSP genes can provide valuable insights into how gene expression spatial structures change across experimental conditions, helping to understand the underlying biological processes.


### Dependencies

In [ ]:
library(edgeR)
library(DESpace)
library(ggplot2)
library(ggspavis)
library(muSpaData)
library(patchwork)

### Load data

In this demo, we will analyze a Stereo-seq dataset from axolotl brain tissues collected at various stages of regeneration [@Wei2022-Stereo-seq-axolotl], which includes multiple samples (i.e. serial sections) measured under multiple experimental conditions (regeneration stages).

In the original dataset, which is publicly available through the [Spatial Transcript Omics DataBase](https://db.cngb.org/stomics/datasets/STDS0000056/data), there are 5 stages with multiple sections, totaling 16 samples (3 to 4 sections per stage).

For computational reasons, we use 3 stages in this chapter -- 2, 10, and 20 days post injury (DPI) -- and only two sections per stage. The analysis framework uses pre-computed spatial domains generated with `r BiocStyle::Biocpkg("Banksy")` [@Singhal2024-BANKSY] (see @sec-ind-clustering). For preprocessing details of this dataset, see the `r BiocStyle::Biocpkg("muSpaData")` Bioconductor package.

In [ ]:
spe <- Wei22_example()
spatialCoordsNames(spe) <- c("x", "y")
spe

In [ ]:
#| code-fold: true
plotCoords(spe, 
    annotate="Banksy_smooth", 
    in_tissue=NULL,
    x_coord="sdimx", 
    y_coord="sdimy", 
    y_reverse=FALSE,
    sample_id="sample_id") + 
    theme(legend.key.size=unit(0, "lines")) + 
    scale_color_manual(values=unname(pals::trubetskoy()))

## DESpace

Here, we briefly demonstrate how to identify DSP genes using the `r BiocStyle::Biocpkg("DESpace")` [@Cai2024-DESpace] Bioconductor package. For more details, see the [package vignette](https://peicai.github.io/DESpace/articles/DSP.html). In brief, spot- or cell-level counts are aggregated into pseudobulk counts for each sample and domain, and these are compared across domains and conditions. DSP focuses on testing whether (condition x domain) interaction terms in the model are different from zero. In other words, does the expression of a gene change between conditions, but change differently for some domains compared to others?

In [ ]:
# run 'DESpace'
dsp <- dsp_test(spe, 
    sample_col="sample_id", 
    condition_col="condition", 
    cluster_col="Banksy_smooth",
    # return full statistics 
    # from 'edgeR::glmFit/LRT'
    verbose=TRUE) 

In [ ]:
# extract gene-level results
names(dsp_global <- dsp$gene_results)
# count significant DSP genes 
# (at 5% FDR significance level)
table(dsp_global$FDR <= 0.05)

In order to identify the key spatial cluster (domain) where expression changes across conditions, we use the `individual_dsp()` function, which focuses on domain-specific DSP. 

In [ ]:
dsp_clu <- individual_dsp(spe, 
    sample_col="sample_id", 
    condition_col="condition",
    cluster_col="Banksy_smooth") 

In [ ]:
# results for cluster 2
dsp_clu2 <- dsp_clu$`2`
head(dsp_clu2, n=3)

# top DSP gene for cluster 2
top_dsp <- dsp_clu2$gene_id[1]

# get gene symbols from Ensembl identifiers
idx <- match(top_dsp, rowData(spe)$gene_name)
(.gs <- rowData(spe)$gene_id[idx])

## Visualization

DSP genes can be further investigated, for example, by plotting the spatial expression of genes across conditions, or the log2-transformed counts per million of genes across different clusters and conditions.

In [ ]:
#| code-fold: true
.ids <- levels(spe$sample_id)
lapply(seq_along(.ids), \(.) {
    .spe <- spe[, spe$sample_id == .ids[.]]
    p <- FeaturePlot(.spe, 
        feature=top_dsp,
        platform="Stereo-seq",
        coordinates=c("sdimx", "sdimy"),
        cluster_col="Banksy_smooth", cluster="2",
        diverging=TRUE, low="gray95", high="blue")
    # ignore alignment
    free(p[[1]] + ggtitle(.ids[.])) 
}) |> wrap_plots(ncol=3)

The boxplots below show the average log-CPM for cluster 2 and for all other clusters (excluding cluster 2) across different time points. The largest difference between cluster 2 and the remaining clusters is observed at 2 DPI, with the difference decreasing over time. At 20 DPI, no significant log-CPM differences are observed between the clusters. This suggests that spatial patterns change across conditions (i.e. time stages).

In [ ]:
#| code-fold: true
# calculate log-CPM
log_cpm <- edgeR::cpm(dsp$estimated_y, log=TRUE)
nms <- colnames(log_cpm)
# wrangling
dat <- data.frame(
    log_cpm=log_cpm[top_dsp, ],
    Banksy=factor(sub(".*_", "", nms)), 
    sample_id=sub("(_[0-9]+)$", "", nms),
    day=as.numeric(sub("([0-9]+)DPI.*", "\\1", nms)))
# visualization
ggplot(dat, aes(factor(day), log_cpm)) + 
    geom_jitter(aes(col=Banksy), size=2, width=0.1) + 
    geom_boxplot(aes(fill=ifelse(Banksy == "2", "cluster 2", "other"))) + 
    scale_x_discrete("Days post injury", breaks=unique(dat$day)) + 
    scale_fill_manual(NULL, values=c("limegreen", "gray")) + 
    labs(title=.gs, y="log2 counts per million (logCPM)") + 
    theme(legend.position="right")

## Appendix

### References {.unnumbered}